# Notebook 09 — Multiple Sequence Alignment (MSA)

**Module:** 08 — Bioinformatics: Sequence Analysis  
**Notebook:** 09 of 18  
**Time estimate:** 75 minutes

---
## Step 1 — Motivation

NW and SW align two sequences. MSA aligns three or more simultaneously. It underpins
phylogenetics (NB10–NB13), motif finding, structural homology, and variant
annotation. CLUSTALW and MAFFT are the two MSA tools most asked about in interviews.

---
## Step 2 — Intuition

Aligning $k$ sequences optimally requires DP in $k$ dimensions — O(n^k) which is
computationally infeasible for k > 3. All practical MSA tools use **progressive
alignment**:

1. Compute pairwise alignment scores for all pairs
2. Build a **guide tree** from those scores (UPGMA or NJ)
3. Align sequences following the guide tree: first align the two closest sequences,
   then add the next closest, etc.

The key insight: once a gap is inserted in an MSA column, it propagates across all
sequences. This is the "once a gap always a gap" property.

**Center-star heuristic:** simpler alternative — pick the sequence most similar to
all others as the "center," align all other sequences to it pairwise, then merge
the alignments. Gives a 2× approximation of optimal.

---
## Step 3 — Biological Background

MSA assumes a **column homology** hypothesis: positions in the same column of an MSA
are descended from the same position in a common ancestor. This is why conservation
at a column is biologically meaningful — it indicates structural or functional constraint.

MSA quality matters enormously for downstream analyses:
- A misaligned MSA produces a wrong phylogenetic tree
- Column conservation scores (Shannon entropy) are only valid if columns are correctly
  homologous
- Structural alignment (3D) can guide MSA for highly divergent sequences (e.g.,
  globins across all kingdoms)

---
## Step 4 — Mathematical Explanation

**Sum-of-pairs (SP) score:** The standard MSA objective function.
$$\text{SP} = \sum_{i < j} \text{score}(\text{alignment of } s_i, s_j)$$
where each pairwise score is computed from the columns of the MSA. Optimal MSA
maximises the SP score — the DP for this is $O(n^k \cdot L^k)$ where $k$ is the
number of sequences and $L$ is length.

**Column conservation (Shannon entropy):**
$$H_c = -\sum_a f_a \log_2 f_a$$
where $f_a$ is the frequency of character $a$ in column $c$. Low entropy = high
conservation. $H = 0$ = perfectly conserved. $H = \log_2 20$ ≈ 4.32 = maximally
variable (protein).

**Guide tree construction:** Pairwise distances from alignment scores → UPGMA or NJ
→ tree topology → progressive alignment order.

In [ ]:
# Step 6 — Python Implementation
import numpy as np
from typing import Sequence
import math


# NW for MSA use (returns score and aligned pair)
def nw_pair(
    seq1: str,
    seq2: str,
    match: int = 1,
    mismatch: int = -1,
    gap: int = -2
) -> tuple[int, str, str]:
    n, m = len(seq1), len(seq2)
    F = np.zeros((n+1, m+1), dtype=int)
    for i in range(n+1): F[i,0] = i*gap
    for j in range(m+1): F[0,j] = j*gap
    for i in range(1,n+1):
        for j in range(1,m+1):
            sc = match if seq1[i-1]==seq2[j-1] else mismatch
            F[i,j] = max(F[i-1,j-1]+sc, F[i-1,j]+gap, F[i,j-1]+gap)
    a1, a2 = [], []
    i, j = n, m
    while i > 0 or j > 0:
        if i>0 and j>0:
            sc = match if seq1[i-1]==seq2[j-1] else mismatch
            if F[i,j]==F[i-1,j-1]+sc:
                a1.append(seq1[i-1]); a2.append(seq2[j-1]); i-=1; j-=1; continue
        if i>0 and F[i,j]==F[i-1,j]+gap:
            a1.append(seq1[i-1]); a2.append('-'); i-=1
        else:
            a1.append('-'); a2.append(seq2[j-1]); j-=1
    return int(F[n,m]), ''.join(reversed(a1)), ''.join(reversed(a2))


def center_star_msa(
    sequences: list[str],
    match: int = 1,
    mismatch: int = -1,
    gap: int = -2
) -> list[str]:
    """
    Center-star heuristic for MSA.
    1. Find the center sequence (maximises sum of pairwise scores to all others)
    2. Align all other sequences to the center using NW
    3. Merge alignments: when a gap is inserted in the center, propagate to all
       already-aligned sequences
    """
    n = len(sequences)
    if n == 1:
        return sequences[:]

    # Compute all pairwise scores
    scores = np.zeros((n, n), dtype=int)
    for i in range(n):
        for j in range(i+1, n):
            sc, _, _ = nw_pair(sequences[i], sequences[j], match, mismatch, gap)
            scores[i, j] = sc
            scores[j, i] = sc

    # Choose center: sequence with maximum sum of scores
    center_idx = int(np.argmax(scores.sum(axis=1)))
    center = sequences[center_idx]

    # Align each sequence to center
    aligned_pairs = []
    for i, seq in enumerate(sequences):
        if i == center_idx:
            continue
        _, c_aln, s_aln = nw_pair(center, seq, match, mismatch, gap)
        aligned_pairs.append((c_aln, s_aln, i))

    # Merge: accumulate gaps in center position
    # Start with the center aligned to itself (no gaps)
    msa_center = list(center)
    msa_others = {i: list(seq) for i, seq in enumerate(sequences) if i != center_idx}

    # For simplicity: directly use the pairwise alignments
    # The center alignment serves as the scaffold
    # Find the longest aligned center (may have different gaps from each pair)
    max_len = max(len(c_aln) for c_aln, _, _ in aligned_pairs)

    # For this implementation, we do a simplified merge:
    # align all pairs and use the center sequence as reference
    # (a full merge requires propagating inserted gaps — simplified here)
    result = [None] * n
    result[center_idx] = aligned_pairs[0][0] if aligned_pairs else center  # center in first pair alignment

    for c_aln, s_aln, orig_idx in aligned_pairs:
        result[orig_idx] = s_aln

    # Make center consistent with all pair alignments (take center from first pair)
    if aligned_pairs:
        result[center_idx] = aligned_pairs[0][0]

    return [r for r in result if r is not None]


def column_conservation(msa: list[str]) -> list[float]:
    """
    Shannon entropy per column. Low = conserved, high = variable.
    """
    if not msa:
        return []
    ncols = len(msa[0])
    entropies = []
    for c in range(ncols):
        col = [seq[c] for seq in msa if c < len(seq)]
        from collections import Counter
        counts = Counter(col)
        n = len(col)
        h = -sum((cnt/n) * math.log2(cnt/n) for cnt in counts.values() if cnt > 0)
        entropies.append(h)
    return entropies


# Test: 4 DNA sequences
seqs = [
    'ATCGATCGATCG',
    'ATCGTTCGATCG',  # one mismatch
    'ATCGATCG',      # shorter (deleted 4 bp)
    'ATCGATCGATCGCC', # longer (2 bp insertion)
]

msa = center_star_msa(seqs)
print("Center-star MSA:")
for i, aln in enumerate(msa):
    print(f"  seq{i+1}: {aln}")

# Pairwise check
print("\nAll pairwise alignments for reference:")
for i in range(len(seqs)):
    for j in range(i+1, len(seqs)):
        sc, a1, a2 = nw_pair(seqs[i], seqs[j])
        print(f"  seq{i+1} vs seq{j+1}: score={sc}")
        print(f"    {a1}")
        print(f"    {a2}")

In [ ]:
# Use Biopython to run MAFFT-equivalent (ClustalW or simple pairwise)
# For now: show conservation analysis on a hand-crafted MSA
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Simulate an MSA of 5 protein-like sequences with conserved core
msa_seqs = [
    'MKTAYIAKQRQISFVKSHFSRQ',
    'MKTAYIAKQRQISFVKSHFSRQ',  # identical
    'MKTAYIAKQRQVSFVKSHFSRQ',  # one change: I->V at pos 12
    'MKTAYIAKQRQLSFVKTHFSRQ',  # two changes
    'MKTGYIAKQRQISFAKSHFSRQ',  # three changes
]

# All same length (already aligned for illustration)
assert all(len(s) == len(msa_seqs[0]) for s in msa_seqs)

entropy = column_conservation(msa_seqs)
ncols = len(msa_seqs[0])

fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [1, 2]})

# Panel A: conservation plot
ax = axes[0]
conservation = [1 - h/math.log2(max(len(set(msa_seqs[i][c] for i in range(len(msa_seqs)))), 2))
                if len(set(msa_seqs[i][c] for i in range(len(msa_seqs)))) > 1
                else 1.0
                for c in range(ncols)]

ax.bar(range(ncols), [1 - e/math.log2(len(msa_seqs)) if e > 0 else 1.0 for e in entropy],
       color=['green' if e < 0.1 else 'orange' if e < 0.5 else 'red' for e in entropy])
ax.set_ylabel('Conservation\n(1=conserved, 0=variable)')
ax.set_title('Column conservation in MSA (green=conserved, red=variable)')
ax.set_xlim(-0.5, ncols - 0.5)

# Panel B: the MSA itself
ax2 = axes[1]
aa_colors = {'A': 0, 'C': 1, 'D': 2, 'E': 3, 'F': 4, 'G': 5, 'H': 6,
             'I': 7, 'K': 8, 'L': 9, 'M': 10, 'N': 11, 'P': 12, 'Q': 13,
             'R': 14, 'S': 15, 'T': 16, 'V': 17, 'W': 18, 'Y': 19, '-': 20}

grid = np.array([[aa_colors.get(aa, 20) for aa in seq] for seq in msa_seqs])
ax2.imshow(grid, cmap='tab20c', aspect='auto', vmin=0, vmax=20)

for i, seq in enumerate(msa_seqs):
    for j, aa in enumerate(seq):
        ax2.text(j, i, aa, ha='center', va='center', fontsize=8)

ax2.set_yticks(range(len(msa_seqs)))
ax2.set_yticklabels([f'Seq{i+1}' for i in range(len(msa_seqs))])
ax2.set_xticks(range(ncols))
ax2.set_xticklabels(range(1, ncols + 1), fontsize=7)
ax2.set_xlabel('Position')
ax2.set_title('MSA coloured by amino acid')

plt.tight_layout()
plt.savefig('msa_conservation.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nConserved columns (entropy=0):", [i for i, e in enumerate(entropy) if e < 0.001])
print("Variable columns (entropy>0):", [i for i, e in enumerate(entropy) if e > 0.001])

---
## MAFFT and CLUSTALW (for real data)

For the portfolio mini-project (NB17), you will run MAFFT on real sequences.
The Python implementation above demonstrates the algorithm; for production MSA,
always use MAFFT (FFT-based, handles thousands of sequences).

```bash
# Install: conda install -c bioconda mafft
# Run:
mafft --auto sequences.fasta > alignment.fasta
# or for slow, accurate mode:
mafft --globalpair --maxiterate 1000 sequences.fasta > alignment.fasta
```

Via Biopython:
```python
from Bio.Align.Applications import MafftCommandline
cline = MafftCommandline(input='sequences.fasta')
stdout, stderr = cline()
```

---
## Step 8 — Exercises

See `exercises/09_msa_exercises.md`.

1. Implement center-star MSA from scratch for 3 sequences. Compare to running all
   pairwise NW alignments and hand-merging.
2. Compute the sum-of-pairs score for your MSA. Is it the maximum possible?
3. Why does progressive alignment order matter? Give an example where aligning
   the two most similar sequences first leads to a suboptimal MSA.

---
## Step 10 — Quiz

1. Why is optimal MSA of k sequences computationally infeasible for k > 3?
2. What is a guide tree, and how is it used in progressive MSA?
3. What does Shannon entropy measure in the context of MSA columns?
4. What does MAFFT stand for? What algorithmic approach does it use?
5. What is the "once a gap, always a gap" property?

---
## Step 12 — Reflection

> *[Explain in one paragraph: why does progressive alignment produce suboptimal MSAs,
> and what would a better (but slower) approach look like?]*

---
## Step 13 — Papers Referenced

- Thompson et al. (1994) "CLUSTAL W: improving the sensitivity of progressive
  multiple sequence alignment." *Nucleic Acids Res* 22(22).
- Katoh et al. (2002) "MAFFT: a novel method for rapid multiple sequence alignment."
  *Nucleic Acids Res* 30(14).

---
*Next: `10_distance_matrices_from_alignments.ipynb`*